# Обучаемые функции

1. Выбрать табличный набор данных для задачи классификации:
    - В котором есть категориальные (не менее двух), и числовые признаки.
    - Необходимо предобработать его: заполнить пропуски, нормализовать и т.д.. Разбить на тренировочную и тестовую часть.
    - Даже если набор данных будет для бинарной классификации, реализация не должна специализироваться под это.
2. Реализовать стандартное обучаемое преобразование: умножение на матрицу, добавление вектора и применение функции активации.
    - Преобразование должно пересчитывать градиент для входа и параметров.
    - Должны поддерживаться не менее трёх функций активации (кроме `Sigmoid`, `Tanh`, `ReLU` и `Softmax`).
3. Реализовать модификацию матричного обучаемого преобразования, в котором скалярное произведение заменено на RBF ядро из SVM.
    - То есть, каждая координата вектора выхода получается как расстояние вектора входа с соответствующим вектором из матрицы параметров, а затем к нему применяется гауссова функция.
    - Для данного преобразования также необходимо реализовать пересчёт градиента.
4. Реализовать обёртку над преобразованиями, которая будет добавлять остаточную связь.
5. Реализовать функцию ошибки `SoftArgMaxCrossEntropy`, которая будет стабильно применять к вектору композицию `SoftArgMax` преобразования и логарифма, для вычисления перекрёстной энтропии с целевым вектором. Данная функция также должна вычислять градиент.
6. Реализовать любую модификацию адаптивного градиентного спуска.
7. Обучить несколько моделей с разным числом стандартных матричных преобразований. 
    - Для каждой модели подобрать хорошие гиперпараметры.
    - Проверить влияние остаточных связей, добавления слоёв батч-нормализации и дропаута.
    - На одном графике - соответствующие кривые обучения.
    - Замерить качество классификации (точность или F-меру) на тестовой части, построить зависимость от числа параметров и преобразований.
8. Повторить предыдущий шаг для модифицированного матричного преобразования.
9. Построить модель из комбинации обычных и модифицированных матричных преобразований. Повторить седьмой шаг для них.


In [1]:
# Python's own stuff
import sys
import math
import dataclasses
import time
import warnings
import itertools as itt
import operator as op
from pathlib import Path
from dataclasses import dataclass, field
from dataclasses import asdict as dc_asdict

# Numpy/Scipy/other
import numpy as np
import scipy.special as sp_spec
import scipy.stats as sp_stats
import pickle

# sklearn components
import sklearn as skl
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, pairwise
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel

# Visualisation
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from ipywidgets import interactive, IntSlider, FloatSlider
from IPython.display import display, Markdown

# Type checking
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from typing import Callable, Optional, Sequence
    from collections.abc import Iterable
    from numpy.typing import NDArray

# Config
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["axes.grid"] = True
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 100
sns.set_style("whitegrid")

# Versions
print(f"Python={sys.version_info}")
print(f"numpy={np.__version__}  sklearn={skl.__version__} matplotlib={mpl.__version__}  pandas={pd.__version__}")


Python=sys.version_info(major=3, minor=14, micro=3, releaselevel='final', serial=0)
numpy=2.5.3  sklearn=1.9.1 matplotlib=3.11.2  pandas=3.0.6


Датасет:

In [2]:
def fetch_ucirepo_cached(dataset_id: int, cache_dir: str | Path = "../../data_cache"):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    cache_path = cache_dir / f"uci_dataset_{dataset_id}.pkl"

    if cache_path.is_file():
        with cache_path.open("rb") as f:
            return pickle.load(f)
    else:
        from ucimlrepo import fetch_ucirepo
        dataset = fetch_ucirepo(id=dataset_id)

        with cache_path.open("wb") as f:
            pickle.dump(dataset, f)

        return dataset

# fetch dataset 
DATASET = fetch_ucirepo_cached(30)

# data (as pandas dataframes) 
X = DATASET.data.features
y = DATASET.data.targets

# metadata 
print("Название:", DATASET.metadata["name"])
print("Описание:", DATASET.metadata["additional_info"]["summary"], sep="\n")
print("Задачи:", DATASET.metadata["tasks"])
print("Переменные:", DATASET.metadata["additional_info"]["variable_info"], sep="\n")

# variable information 
print(DATASET.variables) 


Название: Contraceptive Method Choice
Описание:
This dataset is a subset of the 1987 National Indonesia Contraceptive Prevalence Survey. The samples are married women who were either not pregnant or do not know if they were at the time of interview. The problem is to predict the current contraceptive method choice (no use, long-term methods, or short-term methods) of a woman based on her demographic and socio-economic characteristics.
Задачи: ['Classification']
Переменные:
   1. Wife's age                     (numerical)
   2. Wife's education               (categorical)      1=low, 2, 3, 4=high
   3. Husband's education            (categorical)      1=low, 2, 3, 4=high
   4. Number of children ever born   (numerical)
   5. Wife's religion                (binary)           0=Non-Islam, 1=Islam
   6. Wife's now working?            (binary)           0=Yes, 1=No
   7. Husband's occupation           (categorical)      1, 2, 3, 4
   8. Standard-of-living index       (categorical)      1=lo

Итого - подходит:
- 2 численных признака
- 7 категориальных (только 3 бинарных)
- 3 категории ответа
- пропусков нет